# ML-10 — Content Action Playbook

This playbook converts the validated ML result from W05 into a practical, human-reviewable
decision-support tool. It ranks every page for a human SEO reviewer to inspect first, based
on observed March 2026 search visibility patterns.

**Central intended-use statement:** "The playbook ranks pages for a human SEO reviewer to
inspect first, based on observed March 2026 search visibility patterns. The model identifies
where to look; the human decides what to do."

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup and data loading

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is not set."

print("HF token loaded successfully.")

D:\download_99\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\download_99\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


HF token loaded successfully.


In [2]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March 2026 rows:", len(march_df))
print("Columns:", list(march_df.columns))

March 2026 rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [3]:
page_features = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_gsc_impressions=("gsc_impressions", "sum"),
        march_gsc_clicks=("gsc_clicks", "sum"),
        _pos_count=("gsc_avg_position", "count"),
        _pos_sum=("gsc_avg_position", lambda x: x[x > 0].sum()),
        _pos_valid_n=("gsc_avg_position", lambda x: (x > 0).sum()),
    )
)

page_features["march_gsc_avg_position"] = np.where(
    page_features["_pos_valid_n"] > 0,
    page_features["_pos_sum"] / page_features["_pos_valid_n"],
    np.nan,
)
page_features = page_features.drop(columns=["_pos_count", "_pos_sum", "_pos_valid_n"])

page_features["opportunity_proxy"] = (
    (page_features["march_gsc_impressions"] > 0)
    & (page_features["march_gsc_clicks"] == 0)
).astype(int)

print("Page-level records:", len(page_features))
print("Unique clients:", page_features["client_hash_id"].nunique())
print(f"Opportunity proxy base rate: {page_features['opportunity_proxy'].mean():.1%}")
print(f"Pages with position data: {page_features['march_gsc_avg_position'].notna().mean():.1%}")

Page-level records: 331437
Unique clients: 55
Opportunity proxy base rate: 32.6%
Pages with position data: 52.9%


## 1. Ranked actions + reason codes

The queue: what to do first, and why, in words a human trusts.

### Data and model

- **Population:** 331,437 page/client observations, 55 clients, March 2026 only.
- **Features:** `march_gsc_impressions`, `march_gsc_avg_position` (same as W05).
- **Model:** Logistic Regression trained on the **full** March dataset (not cross-validated).
- **Scoring:** `lr_score = model.predict_proba(X_scaled)[:, 1]`
- **Terminology:** This is an "LR ranking score" or "predicted proxy probability" — NOT "probability that this page needs a refresh."
- **Ranking:** Descending by `lr_score`, with deterministic tie-breaking: `lr_score` desc, `client_hash_id` asc, `content_hash_id` asc.

**Important:** This is a full-data/in-sample scoring output. W05 GroupKFold results remain the evidence for expected out-of-client-fold performance. We do NOT report W07 in-sample Precision@K as validation performance.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
feature_cols = ["march_gsc_impressions", "march_gsc_avg_position"]

# Retrain LR on FULL March dataset for the final W07 queue
scaler_full = StandardScaler()
X_full = page_features[feature_cols].fillna(0).values
X_full_scaled = scaler_full.fit_transform(X_full)

lr_full = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_full.fit(X_full_scaled, page_features["opportunity_proxy"].values)

page_features["lr_score"] = lr_full.predict_proba(X_full_scaled)[:, 1]

print("LR model trained on full March dataset.")
print(f"LR coefficients: impressions={lr_full.coef_[0][0]:+.3f}, position={lr_full.coef_[0][1]:+.3f}")
print(f"LR intercept: {lr_full.intercept_[0]:+.3f}")
print(f"lr_score range: [{page_features['lr_score'].min():.4f}, {page_features['lr_score'].max():.4f}]")

LR model trained on full March dataset.
LR coefficients: impressions=-4.912, position=+2.815
LR intercept: -1.033
lr_score range: [0.0000, 1.0000]


In [5]:
# W04 baseline scoring (reused, not invented)
def position_score_baseline(pos):
    if pd.isna(pos):
        return 0
    if pos <= 3:
        return 0
    elif pos <= 10:
        return 30
    elif pos <= 20:
        return 60
    elif pos <= 50:
        return 80
    else:
        return 100

def impression_score_baseline(imp):
    if imp == 0:
        return 0
    elif imp <= 200:
        return 10
    elif imp <= 1000:
        return 20
    else:
        return 30

def baseline_score(row):
    return position_score_baseline(row["march_gsc_avg_position"]) + impression_score_baseline(row["march_gsc_impressions"])

page_features["baseline_score"] = page_features.apply(baseline_score, axis=1)

print("Baseline scores computed (reused from W04).")
print(f"Baseline score range: [{page_features['baseline_score'].min()}, {page_features['baseline_score'].max()}]")

Baseline scores computed (reused from W04).
Baseline score range: [0, 130]


In [6]:
# Reason code taxonomy (reused/adapted from W04)
def assign_reason_code(row):
    """Assign exactly one reason code per row based on observed March signals.
    The reason code provides a human-readable description of the observed search
    signals associated with the recommendation. It does NOT explain the LR model
    mathematically."""
    pos = row["march_gsc_avg_position"]
    imp = row["march_gsc_impressions"]

    if pd.isna(pos):
        return "position_unavailable"
    if imp == 0:
        return "insufficient_visibility"
    if pos <= 10:
        return "good_position"
    if pos <= 20:
        if imp > 200:
            return "mid_position_moderate_visibility"
        return "mid_position_low_visibility"
    if pos <= 50:
        if imp > 1000:
            return "moderate_position_high_visibility"
        if imp > 200:
            return "moderate_position_moderate_visibility"
        return "moderate_position_low_visibility"
    # pos > 50
    if imp > 1000:
        return "poor_position_high_visibility"
    if imp > 200:
        return "poor_position_moderate_visibility"
    return "poor_position_low_visibility"

page_features["reason_code"] = page_features.apply(assign_reason_code, axis=1)

print("Reason codes assigned.")
print(page_features["reason_code"].value_counts())

Reason codes assigned.
reason_code
position_unavailable                     156133
good_position                             94755
moderate_position_low_visibility          18763
mid_position_moderate_visibility          17865
mid_position_low_visibility               14683
poor_position_low_visibility              11272
moderate_position_moderate_visibility      8382
moderate_position_high_visibility          7638
poor_position_moderate_visibility          1709
poor_position_high_visibility               237
Name: count, dtype: int64


In [7]:
# Rank: lr_score descending, then client_hash_id ascending, then content_hash_id ascending
page_features = page_features.sort_values(
    ["lr_score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True]
).reset_index(drop=True)

page_features["rank"] = range(1, len(page_features) + 1)

print(f"Ranking complete. Total pages: {len(page_features):,}")
print(f"Ranks are unique: {page_features['rank'].nunique() == len(page_features)}")
print(f"Ranks are contiguous: {page_features['rank'].max() == len(page_features)}")

Ranking complete. Total pages: 331,437
Ranks are unique: True
Ranks are contiguous: True


### 2. Action tiers

Practical human-review tiers based on ranking position:

| Tier | Definition |
|---|---|
| **REVIEW** | Top 1,000 ranked pages — primary human-review queue |
| **MONITOR** | Remaining pages with meaningful observable search signal (impressions > 0 and position available) but outside the primary review queue |
| **DEPRIORITISE** | Pages with insufficient observable signal or pages already performing relatively well |

**Important:** These are operational prioritization labels created by the playbook. They do not imply that MONITOR or DEPRIORITISE pages have no value — they indicate relative priority for human review given limited review capacity.

In [8]:
K_REVIEW = 1000

def assign_action(rank):
    if rank <= K_REVIEW:
        return "REVIEW"
    return None  # filled in below based on signal

# First pass: top-1000 get REVIEW
page_features["action"] = page_features["rank"].apply(
    lambda r: "REVIEW" if r <= K_REVIEW else None
)

# For non-REVIEW pages: MONITOR if they have meaningful signal, DEPRIORITISE otherwise
mask_non_review = page_features["action"].isna()
has_signal = (
    (page_features["march_gsc_impressions"] > 0) &
    (page_features["march_gsc_avg_position"].notna())
)

page_features.loc[mask_non_review & has_signal, "action"] = "MONITOR"
page_features.loc[mask_non_review & ~has_signal, "action"] = "DEPRIORITISE"

print("Action tiers assigned.")
print(page_features["action"].value_counts())

Action tiers assigned.
action
MONITOR         174304
DEPRIORITISE    156133
REVIEW            1000
Name: count, dtype: int64


### 3. Confidence tiers

Data-signal confidence tiers indicating how much observable search information is available to the reviewer. This is NOT statistical model confidence.

| Tier | Rule |
|---|---|
| **HIGH** | impressions > 200 AND position available |
| **MODERATE** | (impressions 1–200 AND position available) OR (impressions > 200 AND position unavailable) |
| **LOW** | impressions = 0 OR position unavailable |

Rules are mutually exclusive and logically ordered.

In [9]:
def assign_confidence(row):
    """Data-signal confidence: how much observable search information is available."""
    imp = row["march_gsc_impressions"]
    pos = row["march_gsc_avg_position"]
    pos_available = pd.notna(pos)

    if imp > 200 and pos_available:
        return "HIGH"
    if (1 <= imp <= 200 and pos_available) or (imp > 200 and not pos_available):
        return "MODERATE"
    return "LOW"

page_features["confidence"] = page_features.apply(assign_confidence, axis=1)

print("Confidence tiers assigned.")
print(page_features["confidence"].value_counts())
print(f"\nTotal rows: {len(page_features):,}")
print(f"Sum of tiers: {page_features['confidence'].value_counts().sum():,}")

Confidence tiers assigned.
confidence
LOW         156133
MODERATE     90595
HIGH         84709
Name: count, dtype: int64

Total rows: 331,437
Sum of tiers: 331,437


### 4. Behavioral archetypes

Archetypes help the human reviewer understand why a page appears in the queue. Rules are implemented in a deliberate priority order to ensure mutual exclusivity.

| Archetype | Rule | Recommended human review direction |
|---|---|---|
| **Hidden gems** | impressions > 0, clicks == 0, position > 10 | Investigate whether content alignment or title/meta improvements could convert impressions to clicks |
| **Deep rankers** | position > 50, impressions > 0 | Investigate whether the page targets competitive queries or has structural SEO issues |
| **Emerging pages** | impressions 1–200, position 11–50 | Monitor for growth; investigate if topic targeting is on-track |
| **Already performing** | position <= 10, clicks > 0 | Low priority for refresh; monitor for drift |
| **Ghost pages** | impressions == 0 | Investigate whether page is indexed; check for crawlability or technical issues |
| **Data gaps** | position missing, impressions > 0 | Investigate whether position data is available in other tools; page may be new or recently changed |

In [10]:
def assign_archetype(row):
    """Assign exactly one behavioral archetype. Priority order ensures mutual exclusivity."""
    imp = row["march_gsc_impressions"]
    clicks = row["march_gsc_clicks"]
    pos = row["march_gsc_avg_position"]

    # Ghost pages: no impressions at all
    if imp == 0:
        return "Ghost pages"

    # Data gaps: impressions > 0 but position missing
    if pd.isna(pos):
        return "Data gaps"

    # Already performing: good position and getting clicks
    if pos <= 10 and clicks > 0:
        return "Already performing"

    # Hidden gems: impressions, no clicks, position > 10
    if clicks == 0 and pos > 10:
        return "Hidden gems"

    # Deep rankers: very poor position, some impressions
    if pos > 50 and imp > 0:
        return "Deep rankers"

    # Emerging pages: low-moderate impressions, middling position
    if 1 <= imp <= 200 and 11 <= pos <= 50:
        return "Emerging pages"

    # Remaining: classify by position
    if pos > 50:
        return "Deep rankers"
    if 11 <= pos <= 50:
        return "Emerging pages"
    return "Already performing"

page_features["archetype"] = page_features.apply(assign_archetype, axis=1)

print("Archetypes assigned.")
archetype_counts = page_features["archetype"].value_counts()
archetype_pct = (archetype_counts / len(page_features) * 100).round(1)
archetype_summary = pd.DataFrame({"count": archetype_counts, "pct": archetype_pct})
print(archetype_summary)

Archetypes assigned.
                     count   pct
archetype                       
Ghost pages         154699  46.7
Already performing   96782  29.2
Hidden gems          56049  16.9
Emerging pages       21867   6.6
Data gaps             1434   0.4
Deep rankers           606   0.2


In [11]:
# Archetype examples
print("Example rows per archetype:")
print("=" * 80)
for arch in page_features["archetype"].unique():
    subset = page_features[page_features["archetype"] == arch].head(2)
    print(f"\n--- {arch} ---")
    for _, row in subset.iterrows():
        pos_str = f"{row['march_gsc_avg_position']:.1f}" if pd.notna(row['march_gsc_avg_position']) else 'NaN'
        print(f"  rank={row['rank']}, lr_score={row['lr_score']:.4f}, "
              f"imp={row['march_gsc_impressions']}, clicks={row['march_gsc_clicks']}, "
              f"pos={pos_str}, action={row['action']}, confidence={row['confidence']}")

Example rows per archetype:

--- Hidden gems ---
  rank=1, lr_score=1.0000, imp=1, clicks=0, pos=286.0, action=REVIEW, confidence=MODERATE
  rank=2, lr_score=1.0000, imp=1, clicks=0, pos=244.0, action=REVIEW, confidence=MODERATE

--- Deep rankers ---
  rank=19, lr_score=1.0000, imp=2, clicks=1, pos=257.0, action=REVIEW, confidence=MODERATE
  rank=107, lr_score=1.0000, imp=8, clicks=1, pos=101.4, action=REVIEW, confidence=MODERATE

--- Emerging pages ---
  rank=12827, lr_score=0.9993, imp=21, clicks=1, pos=49.9, action=MONITOR, confidence=MODERATE
  rank=12974, lr_score=0.9992, imp=6, clicks=1, pos=49.5, action=MONITOR, confidence=MODERATE

--- Already performing ---
  rank=60301, lr_score=0.5781, imp=19, clicks=2, pos=10.9, action=MONITOR, confidence=MODERATE
  rank=60442, lr_score=0.5760, imp=10, clicks=1, pos=10.8, action=MONITOR, confidence=MODERATE

--- Ghost pages ---
  rank=134675, lr_score=0.1667, imp=0, clicks=0, pos=NaN, action=DEPRIORITISE, confidence=LOW
  rank=134676, lr_sc

## 2. Intended use and limits

### What the playbook is

The playbook ranks pages for a human SEO reviewer to inspect first, based on observed
March 2026 search visibility patterns. The model identifies where to look; the human decides
what to do.

### What the playbook is NOT

- An autonomous content optimizer
- A content-quality classifier
- A causal model
- A guaranteed business-impact model
- A production monitoring system

### Limitations

| # | Limitation | Why it matters |
|---|---|---|
| 1 | **Proxy-label construction** | The opportunity proxy `(impressions > 0) & (clicks == 0)` is an evaluation definition, not ground truth. Precision@K is partly influenced by this construction. |
| 2 | **Mechanical overlap** | `march_gsc_impressions` is a legitimate decision-time signal but is also mechanically related to the proxy (impressions > 0 is part of the proxy definition). This limits what the metrics demonstrate. |
| 3 | **Single-month data** | Only March 2026 is used. No seasonal, trend, or longitudinal patterns are captured. |
| 4 | **No future/out-of-time validation** | No April+ data exists in this analysis. We cannot assess how the ranking would perform on future data. |
| 5 | **Client heterogeneity** | 55 clients with different industries, content types, and search profiles. Results may not generalize uniformly. |
| 6 | **Client concentration** | One large client dominates ~50% of the top-1000 queue. The queue reflects this client's feature distribution. |
| 7 | **Missing position data** | ~47% of pages lack position data. These are assigned `position_unavailable` and ranked lower. |
| 8 | **Missing/uneven GA4 data** | Only ~4% of daily rows have GA4 data. GA4 features are excluded from the model. |
| 9 | **No query-level GSC data** | The analysis uses page-level aggregates, not the underlying query data. We cannot see which queries drive impressions. |

**Claim language:** We observed patterns in this dataset. We do not claim these patterns will persist, that the model causes improvement, or that the rankings reflect business value.

## 3. Human review + the no-go list

### What the playbook supports

- Prioritizing which pages a human SEO reviewer should inspect first
- Providing human-readable reason codes for why each page appears in the queue
- Indicating data-signal confidence so the reviewer knows how much observable information is available
- Suggesting behavioral archetypes to guide the type of investigation

### What must NOT be automated

The following actions require human judgment and are outside the model's evidence or data scope:

| No-go action | Why it requires human judgment |
|---|---|
| Publishing content | The model has no content-quality signal; it only uses impressions and position. |
| Deleting pages | The model cannot assess whether a page has value beyond search visibility. |
| Merging pages | Requires understanding of content relationships and user intent. |
| Automatically rewriting content | The model has no content-quality signal. |
| Automatically changing title/meta | Requires understanding of the page's content and target queries. |
| Technical SEO changes | Requires crawl analysis, server access, and technical expertise. |
| Claiming that refreshing a page WILL improve rankings | No causal evidence exists. The analysis is cross-sectional. |
| Claiming guaranteed performance/business impact | No business-outcome data is used. No controlled experiment was run. |

### Representative queue examples

For each page in the queue, the correct interpretation is "review this page" rather than
"perform this specific action." The reason code and archetype provide starting context,
but the human reviewer must decide what, if any, action to take.

In [12]:
# Show representative queue examples where the correct interpretation is "review this page"
print("Representative queue examples (correct interpretation: review this page)")
print("=" * 80)

for arch in ["Hidden gems", "Deep rankers", "Ghost pages", "Data gaps"]:
    subset = page_features[page_features["archetype"] == arch].head(1)
    if len(subset) > 0:
        row = subset.iloc[0]
        print(f"\nArchetype: {arch}")
        print(f"  rank={row['rank']}, lr_score={row['lr_score']:.4f}")
        print(f"  impressions={row['march_gsc_impressions']}, clicks={row['march_gsc_clicks']}")
        pos_str2 = f"{row['march_gsc_avg_position']:.1f}" if pd.notna(row['march_gsc_avg_position']) else 'N/A'
        print(f"  position={pos_str2}")
        print(f"  reason_code={row['reason_code']}, confidence={row['confidence']}")
        print(f"  action={row['action']}")
        print(f"  -> Reviewer should investigate: {arch.lower().replace(' ', ' ')}")

Representative queue examples (correct interpretation: review this page)

Archetype: Hidden gems
  rank=1, lr_score=1.0000
  impressions=1, clicks=0
  position=286.0
  reason_code=poor_position_low_visibility, confidence=MODERATE
  action=REVIEW
  -> Reviewer should investigate: hidden gems

Archetype: Deep rankers
  rank=19, lr_score=1.0000
  impressions=2, clicks=1
  position=257.0
  reason_code=poor_position_low_visibility, confidence=MODERATE
  action=REVIEW
  -> Reviewer should investigate: deep rankers

Archetype: Ghost pages
  rank=134675, lr_score=0.1667
  impressions=0, clicks=0
  position=N/A
  reason_code=position_unavailable, confidence=LOW
  action=DEPRIORITISE
  -> Reviewer should investigate: ghost pages

Archetype: Data gaps
  rank=289386, lr_score=0.1665
  impressions=1, clicks=0
  position=N/A
  reason_code=position_unavailable, confidence=LOW
  action=DEPRIORITISE
  -> Reviewer should investigate: data gaps


## 4. Decay / refresh insight

### What we do NOT establish from this dataset

- **We do not establish content decay from this dataset.**
- No longitudinal page-level history exists in this analysis.
- No reliable content creation date exists in the selected feature set.
- The ML warehouse used here contains only March 2026 data.

### Broader cross-sectional observation (qualified)

The broader FlyRank paper reports a cross-sectional age/health pattern where content health
peaks at 61-90 days and declines thereafter. This is an observational finding from a
single point in time, not a controlled experiment. It is subject to compositional
confounding: pages of different ages may differ in content type, topic, and publishing
channel.

W06 already identified the causal limitation: the age-health correlation is observational,
not causal.

**We do NOT claim:**
- "Content decays after 90 days."
- "Pages should be refreshed every 9-12 months."
- "Refreshing causes ranking improvement."

## 5. Monitoring / retrain triggers

This section documents what should be checked if the analysis is refreshed. We do NOT
implement automated monitoring.

| Trigger | Threshold | What to check |
|---|---|---|
| **Performance** | P@100 drops >10 percentage points relative to W05 reference | Re-run GroupKFold validation |
| **Feature drift** | Median/IQR shift >50% for major features | Check if March 2026 feature distributions remain representative |
| **Missingness** | Position availability falls below 40% | Assess whether position data quality is degrading |
| **Client concentration** | Largest client's top-K share exceeds 70% | Check whether the queue is dominated by a single client |
| **Proxy base rate** | Changes by >5 percentage points | Re-evaluate whether the proxy definition remains meaningful |
| **Schema change** | Warehouse grain or feature definitions change | Re-run full pipeline from feature engineering onward |
| **New clients** | Substantial client population change | Re-run grouped validation to assess generalization |

## 6. Cost / value thinking

### The review-capacity problem

331,437 pages cannot realistically be inspected manually. The top-1,000 queue serves as
a primary review batch.

### W05 out-of-client-fold result (validated)

- LR P@1000 = 0.971 +/- 0.030 (W05 GroupKFold)
- Base rate = 32.6%

This indicates much higher proxy-positive concentration in the top-ranked pages than
random selection. Under random selection, we would expect ~326 proxy-positive pages in
1,000. Under the LR ranking, the validated result suggests ~971.

**Important:** We do NOT convert this into revenue, ROI, monetary value, or guaranteed
business impact. We do NOT say "the model saves X hours" unless measured.

## 7. Paper-ready metrics

### Validated results (W05/W06)

These are the out-of-client-fold performance metrics from W05, validated under GroupKFold:

| Method | P@100 | P@500 | P@1000 | P@5000 | Base rate |
|---|---|---|---|---|---|
| Baseline | 0.746 +/- 0.105 | 0.821 +/- 0.124 | 0.828 +/- 0.119 | 0.702 +/- 0.156 | 32.6% |
| Logistic Regression | 0.992 +/- 0.016 | 0.982 +/- 0.025 | 0.971 +/- 0.030 | 0.910 +/- 0.053 | 32.6% |
| Random Forest | 0.964 +/- 0.024 | 0.951 +/- 0.014 | 0.953 +/- 0.011 | 0.948 +/- 0.016 | 32.6% |

### W07 descriptive queue statistics

The following statistics are generated from the full March dataset (in-sample). They describe
the queue composition, not model performance.

In [13]:
# W07 descriptive queue statistics
print("=" * 80)
print("W07 DESCRIPTIVE QUEUE STATISTICS (in-sample, full March dataset)")
print("=" * 80)
print()

print(f"Total pages: {len(page_features):,}")
print(f"Base rate (opportunity_proxy): {page_features['opportunity_proxy'].mean():.1%}")
print(f"Top-1000 queue size: {K_REVIEW}")
print()

print("Action distribution:")
for action, count in page_features["action"].value_counts().items():
    print(f"  {action}: {count:,} ({count/len(page_features)*100:.1f}%)")
print()

print("Archetype distribution:")
for arch, count in page_features["archetype"].value_counts().items():
    print(f"  {arch}: {count:,} ({count/len(page_features)*100:.1f}%)")
print()

print("Confidence distribution:")
for conf, count in page_features["confidence"].value_counts().items():
    print(f"  {conf}: {count:,} ({count/len(page_features)*100:.1f}%)")
print()

# Client concentration in top-1000
top1000 = page_features.head(K_REVIEW)
client_counts_top1000 = top1000["client_hash_id"].value_counts()
print("Client concentration in top-1000:")
for c, n in client_counts_top1000.head(5).items():
    print(f"  {c}: {n} pages ({n/K_REVIEW*100:.1f}%)")
print()

# Overlap with baseline top-1000
baseline_top1000 = page_features.nlargest(K_REVIEW, "baseline_score")
lr_top1000_ids = set(page_features.head(K_REVIEW)["content_hash_id"])
bl_top1000_ids = set(baseline_top1000["content_hash_id"])
overlap = len(lr_top1000_ids & bl_top1000_ids)
print(f"Overlap between LR top-1000 and baseline top-1000: {overlap} pages")
print(f"  ({overlap/K_REVIEW*100:.1f}% of each set)")

W07 DESCRIPTIVE QUEUE STATISTICS (in-sample, full March dataset)

Total pages: 331,437


Base rate (opportunity_proxy): 32.6%
Top-1000 queue size: 1000

Action distribution:
  MONITOR: 174,304 (52.6%)
  DEPRIORITISE: 156,133 (47.1%)
  REVIEW: 1,000 (0.3%)

Archetype distribution:
  Ghost pages: 154,699 (46.7%)
  Already performing: 96,782 (29.2%)
  Hidden gems: 56,049 (16.9%)
  Emerging pages: 21,867 (6.6%)
  Data gaps: 1,434 (0.4%)
  Deep rankers: 606 (0.2%)

Confidence distribution:
  LOW: 156,133 (47.1%)
  MODERATE: 90,595 (27.3%)
  HIGH: 84,709 (25.6%)

Client concentration in top-1000:
  client_08a6a72ff48e62c0: 500 pages (50.0%)
  client_ba65e80a1116ae41: 135 pages (13.5%)
  client_f623b01661d4bfe4: 114 pages (11.4%)
  client_3ffa76342f366962: 56 pages (5.6%)
  client_62f4a7e64f5e0096: 46 pages (4.6%)



Overlap between LR top-1000 and baseline top-1000: 6 pages
  (0.6% of each set)


## 8. Claim integrity audit

Before final export, we classify major claims by their evidence level.

| Claim | Classification |
|---|---|
| Playbook ranks pages for human review | **SUPPORTED** — this is exactly what it does |
| LR outperforms baseline under grouped validation | **SUPPORTED** — W05 GroupKFold shows LR P@K > Baseline P@K |
| Position is an independent signal | **SUPPORTED** — position is not part of the proxy definition; signal audit confirmed |
| Proxy construction limits interpretation | **SUPPORTED** — impressions > 0 is part of the proxy; Precision@K is partly mechanical |
| Model generalizes to unseen clients | **SUPPORTED WITH QUALIFICATION** — W05 GroupKFold shows generalization across 5 client groups, but 55 clients is a limited population |
| Content decays | **DO NOT CLAIM** — no longitudinal data; cross-sectional age pattern is observational |
| Refreshing causes ranking improvement | **DO NOT CLAIM** — no controlled experiment; no intervention data |
| Model predicts pages that need refresh | **DO NOT CLAIM** — the model ranks proxy-positive pages; "needs refresh" is a business judgment |

## 9. Figures

Reusable paper figures saved to `work/figures/`.

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIGURES_DIR = os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..", "..")), "work", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Figures directory: {FIGURES_DIR}")

Matplotlib is building the font cache; this may take a moment.


Figures directory: D:\download_99\Dreams\Interns\flyrank.ai intern\Flyrank_ML_Assign\work\figures


In [15]:
# Figure 1: Action distribution
fig, ax = plt.subplots(figsize=(6, 4))
action_counts = page_features["action"].value_counts()
colors = {"REVIEW": "#2171b5", "MONITOR": "#6baed6", "DEPRIORITISE": "#bdd7e7"}
bars = ax.bar(action_counts.index, action_counts.values,
              color=[colors.get(a, "gray") for a in action_counts.index])
ax.set_ylabel("Number of pages")
ax.set_title("Action tier distribution (full March 2026 dataset)")
for bar, val in zip(bars, action_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "action_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: action_distribution.png")

Saved: action_distribution.png


C:\Users\yousef\AppData\Local\Temp\ipykernel_14208\1417614366.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Figure 2: Archetype breakdown
fig, ax = plt.subplots(figsize=(8, 4))
arch_counts = page_features["archetype"].value_counts()
arch_pct = (arch_counts / len(page_features) * 100).round(1)
bars = ax.barh(arch_counts.index[::-1], arch_counts.values[::-1], color="#2171b5")
ax.set_xlabel("Number of pages")
ax.set_title("Archetype distribution (full March 2026 dataset)")
for bar, pct in zip(bars, arch_pct.values[::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f"{pct}%", ha="left", va="center", fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "archetype_breakdown.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: archetype_breakdown.png")

Saved: archetype_breakdown.png


C:\Users\yousef\AppData\Local\Temp\ipykernel_14208\1593494095.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Figure 3: Client concentration in top-1000
fig, ax = plt.subplots(figsize=(8, 4))
top1000_client = page_features.head(K_REVIEW)["client_hash_id"].value_counts()
n_clients = len(top1000_client)
bars = ax.bar(range(n_clients), top1000_client.values, color="#2171b5")
ax.set_xlabel("Client (ranked by page count)")
ax.set_ylabel("Pages in top-1000")
ax.set_title(f"Client concentration in top-1000 queue ({n_clients} clients)")
ax.axhline(y=K_REVIEW/n_clients, color="red", linestyle="--", alpha=0.5, label="Uniform distribution")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "client_concentration.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: client_concentration.png")
print(f"Top client: {top1000_client.index[0]} ({top1000_client.iloc[0]} pages, {top1000_client.iloc[0]/K_REVIEW*100:.0f}%)")

Saved: client_concentration.png
Top client: client_08a6a72ff48e62c0 (500 pages, 50%)


C:\Users\yousef\AppData\Local\Temp\ipykernel_14208\3015592522.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Figure 4: Confidence distribution
fig, ax = plt.subplots(figsize=(6, 4))
conf_counts = page_features["confidence"].value_counts()
conf_order = ["HIGH", "MODERATE", "LOW"]
conf_vals = [conf_counts.get(c, 0) for c in conf_order]
colors = {"HIGH": "#2ca02c", "MODERATE": "#ff7f0e", "LOW": "#d62728"}
bars = ax.bar(conf_order, conf_vals, color=[colors[c] for c in conf_order])
ax.set_ylabel("Number of pages")
ax.set_title("Data-signal confidence distribution")
for bar, val in zip(bars, conf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "confidence_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confidence_distribution.png")

Saved: confidence_distribution.png


C:\Users\yousef\AppData\Local\Temp\ipykernel_14208\22735695.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Figure 5: Queue vs baseline overlap
fig, ax = plt.subplots(figsize=(6, 4))

# Compute overlap
lr_set = set(page_features.head(K_REVIEW)["content_hash_id"])
bl_set = set(page_features.nlargest(K_REVIEW, "baseline_score")["content_hash_id"])
both = len(lr_set & bl_set)
lr_only = len(lr_set - bl_set)
bl_only = len(bl_set - lr_set)

categories = ["LR only", "Both", "Baseline only"]
values = [lr_only, both, bl_only]
colors = ["#2171b5", "#6a51a3", "#e7969c"]
bars = ax.bar(categories, values, color=colors)
ax.set_ylabel("Number of pages")
ax.set_title(f"Top-1000 queue overlap: LR vs baseline")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{val}", ha="center", va="bottom", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "queue_vs_baseline_overlap.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: queue_vs_baseline_overlap.png")
print(f"Overlap: {both} pages ({both/K_REVIEW*100:.1f}% of each set)")

Saved: queue_vs_baseline_overlap.png
Overlap: 6 pages (0.6% of each set)


C:\Users\yousef\AppData\Local\Temp\ipykernel_14208\1343086520.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Exports

In [20]:
OUTPUTS_DIR = os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..", "..")), "work", "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

export_cols = [
    "client_hash_id", "content_hash_id", "rank", "lr_score", "baseline_score",
    "action", "reason_code", "archetype", "confidence",
    "march_gsc_impressions", "march_gsc_clicks", "march_gsc_avg_position"
]

# Full queue
full_queue = page_features[export_cols].copy()
full_path = os.path.join(OUTPUTS_DIR, "ml_action_queue.csv")
full_queue.to_csv(full_path, index=False)

# Top-1000
top1000_queue = full_queue.head(K_REVIEW).copy()
top1000_path = os.path.join(OUTPUTS_DIR, "ml_action_queue_top1000.csv")
top1000_queue.to_csv(top1000_path, index=False)

print("=" * 80)
print("EXPORT SUMMARY")
print("=" * 80)
print()

print(f"Full queue:")
print(f"  Path: {full_path}")
print(f"  Rows: {len(full_queue):,}")
print(f"  Columns: {len(full_queue.columns)}")
print(f"  Top rank: {full_queue['rank'].min()}")
print(f"  Action counts: {dict(full_queue['action'].value_counts())}")
print(f"  Archetype counts: {dict(full_queue['archetype'].value_counts())}")
print(f"  Confidence counts: {dict(full_queue['confidence'].value_counts())}")
print(f"  Reason code counts: {dict(full_queue['reason_code'].value_counts())}")
print()

print(f"Top-1000 queue:")
print(f"  Path: {top1000_path}")
print(f"  Rows: {len(top1000_queue):,}")
print(f"  Columns: {len(top1000_queue.columns)}")
print(f"  Action counts: {dict(top1000_queue['action'].value_counts())}")
print(f"  Archetype counts: {dict(top1000_queue['archetype'].value_counts())}")
print(f"  Confidence counts: {dict(top1000_queue['confidence'].value_counts())}")
print(f"  Reason code counts: {dict(top1000_queue['reason_code'].value_counts())}")
print()

# Verify no private data
print("Privacy check:")
print(f"  IDs remain pseudonymized: YES (client_hash_id, content_hash_id only)")
print(f"  No names/URLs/private info: YES")

EXPORT SUMMARY

Full queue:
  Path: D:\download_99\Dreams\Interns\flyrank.ai intern\Flyrank_ML_Assign\work\outputs\ml_action_queue.csv
  Rows: 331,437
  Columns: 12
  Top rank: 1
  Action counts: {'MONITOR': 174304, 'DEPRIORITISE': 156133, 'REVIEW': 1000}
  Archetype counts: {'Ghost pages': 154699, 'Already performing': 96782, 'Hidden gems': 56049, 'Emerging pages': 21867, 'Data gaps': 1434, 'Deep rankers': 606}
  Confidence counts: {'LOW': 156133, 'MODERATE': 90595, 'HIGH': 84709}
  Reason code counts: {'position_unavailable': 156133, 'good_position': 94755, 'moderate_position_low_visibility': 18763, 'mid_position_moderate_visibility': 17865, 'mid_position_low_visibility': 14683, 'poor_position_low_visibility': 11272, 'moderate_position_moderate_visibility': 8382, 'moderate_position_high_visibility': 7638, 'poor_position_moderate_visibility': 1709, 'poor_position_high_visibility': 237}

Top-1000 queue:
  Path: D:\download_99\Dreams\Interns\flyrank.ai intern\Flyrank_ML_Assign\work\outp

## Self-check

Before submission, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.